# Evaluation: rules vs LLM prompts against hand labels

## Load labels and predictions

In [15]:
import re
from itertools import combinations
import duckdb
import pandas as pd

labels = pd.read_csv("../data/labels/labels.csv")

con = duckdb.connect("../council.duckdb")
rules = con.sql("SELECT vendor_raw, std_name_rules, category_rules FROM supplier_rules").df()
llm = con.sql("SELECT vendor_raw, std_name_llm_v1, category_llm_v1, is_person_llm_v1 FROM supplier_llm_v1").df()
tables = [t[0] for t in con.sql("SHOW TABLES").fetchall()]
llm2 = con.sql("SELECT vendor_raw, std_name_llm_v2, category_llm_v2, is_person_llm_v2 FROM supplier_llm_v2").df() if "supplier_llm_v2" in tables else None
con.close()

df = labels.merge(rules, on="vendor_raw", how="left").merge(llm, on="vendor_raw", how="left")
if llm2 is not None:
    df = df.merge(llm2, on="vendor_raw", how="left")
VERSIONS = ["v1", "v2"] if llm2 is not None else ["v1"]
print("labelled rows:", len(df), "| missing rule answers:", df["category_rules"].isna().sum(),
      "| missing LLM answers:", df["category_llm_v1"].isna().sum())

labelled rows: 150 | missing rule answers: 0 | missing LLM answers: 0


## Scores
- **category_accuracy**: share of suppliers given the same category as the hand label
- **name_exact_match**: standard name matches the hand label after ignoring case, punctuation and Ltd/PLC
- **duplicate precision / recall**: of the pairs each system treats as the same supplier, how many really are (precision), and of the real duplicate pairs, how many it found (recall)

In [16]:
STOP = {"LTD", "LIMITED", "PLC", "LLP", "CIC", "THE", "CO", "INC", "UK"}

def norm(name):
    s = str(name).upper().replace("&", " AND ")
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    return " ".join(w for w in s.split() if w not in STOP)

def pair_scores(truth, pred):
    pairs = list(combinations(range(len(truth)), 2))
    same_t = {p for p in pairs if truth[p[0]] == truth[p[1]]}
    same_p = {p for p in pairs if pred[p[0]] == pred[p[1]]}
    tp = len(same_t & same_p)
    precision = tp / len(same_p) if same_p else 1.0
    recall = tp / len(same_t) if same_t else 1.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

truth_names = df["true_standard_name"].map(norm).tolist()
results = []
SYSTEMS = [("Rules", "std_name_rules", "category_rules")] + [
    (f"LLM {v}", f"std_name_llm_{v}", f"category_llm_{v}") for v in VERSIONS]
for system, name_col, cat_col in SYSTEMS:
    pred_names = df[name_col].map(norm).tolist()
    p, r, f1 = pair_scores(truth_names, pred_names)
    results.append({
        "system": system,
        "category_accuracy": (df[cat_col] == df["true_category"]).mean(),
        "name_exact_match": sum(a == b for a, b in zip(truth_names, pred_names)) / len(df),
        "duplicate_precision": p,
        "duplicate_recall": r,
        "duplicate_f1": f1,
    })

results = pd.DataFrame(results).set_index("system")
print("true duplicate pairs in sample:", sum(1 for a, b in combinations(truth_names, 2) if a == b))
results.style.format("{:.0%}")

true duplicate pairs in sample: 20


,category_accuracy,name_exact_match,duplicate_precision,duplicate_recall,duplicate_f1
system,,,,,
Rules,56%,72%,100%,60%,75%
LLM v1,82%,78%,80%,100%,89%
LLM v2,85%,80%,100%,65%,79%


## Private individuals

In [17]:
truth_person = df["is_person"].eq("Y")
print("individuals in your labels:", truth_person.sum())
for v in VERSIONS:
    pred_person = df[f"is_person_llm_{v}"].fillna(False).astype(bool)
    print(f"LLM {v} flagged:", pred_person.sum(), "| correct:", (truth_person & pred_person).sum())
print("rules flag individuals: never (no rule for it)")

individuals in your labels: 1
LLM v1 flagged: 1 | correct: 1
LLM v2 flagged: 1 | correct: 1
rules flag individuals: never (no rule for it)


## Category errors (latest prompt)

In [18]:
LATEST = VERSIONS[-1]
errors = df.loc[df[f"category_llm_{LATEST}"] != df["true_category"],
                ["id", "vendor_raw", "expense_type", "true_category", "category_llm_v1"]
                + ([f"category_llm_{LATEST}"] if LATEST != "v1" else []) + ["category_rules", "notes"]]
print(f"LLM {LATEST} category errors: {len(errors)} of {len(df)}")
errors

LLM v2 category errors: 22 of 150


,id,vendor_raw,expense_type,true_category,category_llm_v1,category_llm_v2,category_rules,notes
4,5,ALAN JOHNSTON PARTNERSHIP LTD,"LCC Land,Buildings,Structures-Build Work",Construction & Property,Construction & Property,Professional Services,Professional Services,NaN
14,15,BRAMBLE HUB LIMITED,Consultancy,Professional Services,Professional Services,IT & Telecoms,Professional Services,NaN
16,17,C.E.L.L.S,Misc. Expenses - General,Other,Other,Community & Voluntary,Other,NaN
33,34,EXPANSE GROUP LTD,Voluntary Assoc. 2017/18,Community & Voluntary,Community & Voluntary,Construction & Property,Community & Voluntary,NaN
36,37,GEORGE TRUST HOUSE,Private Contractors (Gen),Other,Construction & Property,Community & Voluntary,Community & Voluntary,NaN
42,43,GREENBANK PROJECT,Room Hire,Social Care,Community & Voluntary,Community & Voluntary,Community & Voluntary,NaN
48,49,HCHQ LTD,Events,"Culture, Leisure & Events","Culture, Leisure & Events",Other,"Culture, Leisure & Events",NaN
52,53,HOMES TOGETHER LTD,Private Contractors (Gen),Housing & Accommodation,Housing & Accommodation,Social Care,Housing & Accommodation,NaN
55,56,Ideas Alliance CIC,Misc. Expenses - General,Community & Voluntary,Community & Voluntary,Professional Services,Other,NaN
56,57,INSTAGRID UK LTD,Miscellaneous - General,Energy & Utilities,Energy & Utilities,Goods & Supplies,Other,NaN


## Which categories get confused

In [19]:
pd.crosstab(errors["true_category"], errors[f"category_llm_{LATEST}"], rownames=["your label"], colnames=[f"LLM {LATEST} said"])

LLM v2 said,Community & Voluntary,Construction & Property,"Culture, Leisure & Events",Goods & Supplies,IT & Telecoms,Other,Professional Services,Social Care,Transport & Travel
your label,,,,,,,,,
Community & Voluntary,0,1,1,0,0,0,3,2,0
Construction & Property,0,0,0,0,0,0,1,0,0
"Culture, Leisure & Events",0,0,0,0,0,1,0,0,0
Education & Childcare,1,0,0,0,0,0,0,0,0
Energy & Utilities,0,0,0,1,0,0,0,0,0
Goods & Supplies,0,0,0,0,0,0,0,0,1
Housing & Accommodation,0,0,0,0,0,0,0,2,0
Other,2,0,0,1,0,0,0,0,0
Professional Services,0,1,0,0,1,0,0,0,0


## Name differences

In [20]:
name_errors = df.loc[[a != b for a, b in zip(truth_names, df[f"std_name_llm_{LATEST}"].map(norm))],
                     ["id", "vendor_raw", "true_standard_name", f"std_name_llm_{LATEST}", "std_name_rules"]]
print(f"LLM {LATEST} names that differ from yours (after removing Ltd/punctuation/case): {len(name_errors)}")
name_errors

LLM v2 names that differ from yours (after removing Ltd/punctuation/case): 30


,id,vendor_raw,true_standard_name,std_name_llm_v2,std_name_rules
1,2,A LEVEL ACCESS (ORMSKIRK) LTD,A Level Access,A Level Access (Ormskirk),A LEVEL ACCESS (ORMSKIRK) LTD
3,4,ACTIVE 8 SUPPORT SERVICES,Active8 Support Services,Active 8 Support Services,ACTIVE 8 SUPPORT SERVICES
5,6,ANFIELD CHILDRENS CENTRE,Anfield Children's Centre,Anfield Childrens Centre,ANFIELD CHILDRENS CENTRE
6,7,Anfield Childrens Centre,Anfield Children's Centre,Anfield Childrens Centre,ANFIELD CHILDRENS CENTRE
11,12,Austin Mitchel Hewitt,Austin Mitchell Hewitt,Austin Mitchel Hewitt,Austin Mitchel Hewitt
12,13,BEECHWOOD SPECIALIST CARE-NURSING H,Beechwood Specialist Care,Beechwood Specialist Care-Nursing Home,BEECHWOOD SPECIALIST CARE-NURSING H
24,25,E4LAW LTD t/a LEXTOX,E4Law,E4Law t/a Lextox,E4LAW LTD t/a LEXTOX
37,38,Gig-mates Ltd,Gigmates,Gig-mates,Gigmates Ltd
38,39,Gigmates Ltd,Gigmates,Gig-mates,Gigmates Ltd
44,45,GSS (NI) LTD T/A PLOTBOX,GSS (NI)(PlotBox),GSS (NI) T/A Plotbox,GSS (NI) LTD T/A PLOTBOX


## Save

In [21]:
import os
os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/eval_rows.csv", index=False)
results.to_csv("../data/processed/eval_results.csv")
print("saved data/processed/eval_rows.csv and eval_results.csv")

saved data/processed/eval_rows.csv and eval_results.csv
